In [1]:
# @title Imports! Impor(s)tant!
import numpy as np
! pip install numpy-financial
import numpy_financial as npf
import matplotlib.pyplot as plt

import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# @title Dictionary Defenitions (press to reset)
substanceDict = {}
costDict = {}
systemVariables = {}

In [ ]:
# @markdown **Global Variables**
# @markdown  -
# @markdown inflation and interest rate
inflation = 0.035 # @param {"type":"slider", min:0, max:0.2, step:0.005}
interest = 0.06 # @param {"type":"slider","min":0,  max:0.2, step:0.005}
# @markdown Beta power sector (negative = volitile, positive = stable)
beta = 0.79 # @param {"type":"slider", min:-1, max:1, step:0.01}
# @markdown Equaty
equityRiskPremium = 0.055 # @param {"type":"slider", min:0, max:5, step:0.01}
equityRatio = 1 #@param {"type":"slider", min:-1, max:1, step:0.1}
# @markdown Project variables
lifetime = 20 #@param (type:"number")
# @markdown Corporate tax
corpTax = 0.258 #@param {type:"slider", min:0, max:1,step:0.01}

systemVariables["inflation"] = inflation
systemVariables["interest"] = interest
systemVariables["beta"] = beta
systemVariables["equityRiskPremium"] = equityRiskPremium
systemVariables["equityRatio"] = equityRatio
systemVariables["lifetime"] = lifetime
systemVariables["corpTax"] = corpTax


In [ ]:
# @markdown Variables

substance = "" # @param {type:"string"}
# @markdown Flow in kg/year
input = None # @param {type:"number"}
output = None # @param {type:"number"}
# @markdown type of substance
substanceType = "unused" # @param ["starting material" ,"unused", "product"]
# @markdown Cost per kg
cost = None # @param {type:"number"}

substanceDict[substance] = {"input": input, "output": output, "net": (input-output), "type":substanceType, "cost": cost}

In [ ]:
# @markdown What is the cost?

source = "" # @param {type:"string"}

# @markdown Type of Cost and amount in Euro
costType = "once" # @param ["once","yearly","yearly per substance"]
costs = None # @param {type:"number"}
perSubstanceKey = '' # @param {type:"string"}

# @markdown Fixed or Variable
costCatagory = "fixed" # @param ["fixed","variable", "other"]

costDict[source] = {"type": costType, "costs": costs, "substanceKey":perSubstanceKey, "category":costCatagory}


In [ ]:
print(substanceDict)
print(costDict)
print(systemVariables)

In [ ]:
# @title Default Settings (Brewery example)
def addDefaultBreweryTest():
  # adding substances
  substanceDict["beer"] = {'input': 0, 'output': 2176200, 'net': -2176200, 'type': 'product', 'cost': 1.22876}
  substanceDict["water"] = {'input': 2067390, 'output': 0, 'net': 2067390, 'type': 'starting material', 'cost': 0.00145}
  substanceDict["barley"] = {'input': 435240, 'output': 0, 'net': 435240, 'type': 'starting material', 'cost': 2.08}
  substanceDict["hop"] = {'input': 4352, 'output': 0, 'net': 4352, 'type': 'starting material', 'cost': 36.344}
  substanceDict["yeast"] = {'input': 1088, 'output': 0, 'net': 1088, 'type': 'starting material', 'cost': 140.217}

  #adding costs (initial costs)
  costDict["basic install costs watersupply"] = {'type': 'once', 'costs': 1708,'substanceKey':'', 'category': 'fixed'}
  costDict["machine costs"] = {'type': 'once', 'costs': 201078,'substanceKey':'', 'category': 'fixed'}
  costDict["initial location costs"] = {'type': 'once', 'costs': 41760,'substanceKey':'', 'category': 'fixed'}
  #adding costs (yearly)
  costDict["cleaner"] = {'type': 'yearly', 'costs': 5184,'substanceKey':'', 'category': 'fixed'}
  costDict["hiring Location"] = {'type': 'yearly', 'costs': 41760,'substanceKey':'', 'category': 'fixed'}
  costDict["total energy price"] = {'type': 'yearly', 'costs': 185660.87,'substanceKey':'', 'category': 'variable'}
  #adding substancedependent costs
  costDict["bottles"] = {'type': 'yearly per substance', 'costs': 0.55,'substanceKey':'beer', 'category': 'variable'}
  costDict["tax on water"] = {'type': 'yearly per substance', 'costs': 0.00043,'substanceKey':'water','category': 'variable'}

  #adding systemvariables
  systemVariables['inflation'] = 0.035
  systemVariables['interest'] = 0.06
  systemVariables['beta'] = 0.79
  systemVariables['equityRiskPremium'] = 0.055
  systemVariables['equityRatio'] = 1
  systemVariables['lifetime'] = 20
  systemVariables['corpTax'] = 0.19


addDefaultBreweryTest()

In [ ]:
# @title Sensitivity Analysis
fig = go.Figure()

#parameters = [('system','inflation'), ('system','interest'), ('system','beta'),
              #('system','equityRiskPremium'), ('substance', 'beer', 'cost'),('substance', 'water', 'cost'),
              #('substance', 'barley', 'cost'),('substance', 'hop', 'cost'),('substance', 'yeast', 'cost'),
              #('cost', 'bottles', 'costs'),('cost', 'cleaner', 'costs'),('cost', 'hiring Location', 'costs'),
              #('cost','total energy price', 'costs')]

parameters = [('system','inflation')]

dictDict = {'substance':substanceDict, 'cost':costDict,'system':systemVariables}
for items in parameters:
  valuelist = []
  npvresults = []
  if len(items) == 2: actualvalue = dictDict[items[0]][items[1]]
  else: actualvalue = dictDict[items[0]][items[1]][items[2]]
  for value in [(actualvalue + actualvalue * (x/100))  for x in range(-20,21)]:
    valuelist.append(value)
    if len(items) == 2: dictDict[items[0]][items[1]] = value
    else: dictDict[items[0]][items[1]][items[2]] = value
    brewery = companyToBeAnalysed(dictDict["substance"], dictDict["cost"], dictDict["system"])
    npvresults.append(brewery.getnpv())
  if len(items) == 2:
    dictDict[items[0]][items[1]] = actualvalue
    fig.add_trace(go.Scatter(x=[f"{x}%" for x in range(-20,21)], y=npvresults, name=items[1]))
  else:
    dictDict[items[0]][items[1]][items[2]] = actualvalue
    fig.add_trace(go.Scatter(x=[f"{x}%" for x in range(-20,21)], y=npvresults, name=f"{items[1]} {items[2]}"))

fig.update_layout(title=dict(text="Sensitivity Analysis"), xaxis=dict(title=dict(text="Percentage change")), yaxis=dict(title=dict(text="NPV")))
fig.show()

In [29]:
# @title Class Creation

class companyToBeAnalysed():
  def __init__(self, substanceDict, costDict, systemVariables):

    #Get the important lists
    self.substanceDict = substanceDict
    self.costDict = costDict

    #unpack some of system variables
    self.interest = systemVariables["interest"]
    self.inflation = systemVariables["inflation"]
    self.beta = systemVariables["beta"]
    self.equityRiskPremium = systemVariables["equityRiskPremium"]
    self.equityRatio = systemVariables["equityRatio"]
    self.corporateTax = systemVariables["corpTax"]
    self.lifetime = systemVariables["lifetime"]

    #Calculate the RFIR and WACC because im sure we're going to need it somewhere
    self.RFIR = ((1 + self.interest)/(1 + self.inflation)) - 1
    self.WACC = ((self.RFIR) + (self.beta * self.equityRiskPremium))

    self.printstuff = systemVariables["printstuff"]

    self.runSim()
    #self.printStuff()
    if self.printstuff:
      self.makeGraph()


  def runSim(self):
    self.fixedCostContainter = []
    self.variableCostContainer = []
    self.revenueContainer = []
    self.operatingIncomeContainer = []
    self.netIncomeContainer = []
    self.discountedCashFlowContainer = []
    self.CumDCFContainer = []

    def calculateBreakeven(self):
      for i in range(len(self.CumDCFContainer)):
        if self.CumDCFContainer[i] >= 0:
          return ((i-1) + ((-self.CumDCFContainer[i-1]) / (self.CumDCFContainer[i]-self.CumDCFContainer[i-1])))
      return None

    def calculateInitialCosts(self):
      cost = 0
      for key in self.costDict.keys():
        if self.costDict[key]["type"] == "once":
          cost += self.costDict[key]["costs"]
      return cost

    def calculateYearlyFixed(self):
      cost = 0

      #Add in other costs
      for key in self.costDict.keys():
        if self.costDict[key]["type"] == "yearly" and self.costDict[key]["category"] == "fixed":
          cost += self.costDict[key]["costs"]
      return cost

    def calculateYearlyCosts(self):
      cost = 0
      for key in self.substanceDict.keys():
        if self.substanceDict[key]["net"] > 0:
          cost += self.substanceDict[key]["net"] * self.substanceDict[key]["cost"]

      materialcost = cost
      #print(f"Material Cost : {materialcost}")

      #Add The yearly variable costs and costs of perSubstance used costs
      for key in self.costDict.keys():
        if self.costDict[key]["type"] == "yearly" and self.costDict[key]["category"] == "variable":
          cost += self.costDict[key]["costs"]
        if self.costDict[key]["type"] == "yearly per substance" and self.costDict[key]["category"] == "variable":
          cost += self.costDict[key]["costs"] * abs(self.substanceDict[self.costDict[key]["substanceKey"]]["net"])
      return cost

    #Basically the same function and i dont like it. Probs could be optimised
    def calculateYearlyRev(self):
      totYearlyrev = 0
      for key in self.substanceDict.keys():
        if self.substanceDict[key]["type"] == "product":
          totYearlyrev += -self.substanceDict[key]["net"] * (self.substanceDict[key]["cost"])
      return totYearlyrev

    def calculateInflation(self, year):
      return 1
      #return ((1 + self.inflation) ** year)


    #A function to set up the initial state. Which is substantially different from the following years
    def setupInitialState(self):
      self.fixedCostContainter.append(-calculateInitialCosts(self)) # Initial cost per capacity * capacity
      self.variableCostContainer.append(0) # 0 because no variable costs yet, factory has not yet been operating
      self.revenueContainer.append(0) # 0 because nothing has been sold yet
      self.operatingIncomeContainer.append(self.fixedCostContainter[-1] + self.variableCostContainer[-1] + self.revenueContainer[-1]) # sum of the above 3 variable
      self.netIncomeContainer.append(self.variableCostContainer[-1] + self.revenueContainer[-1] + self.operatingIncomeContainer[-1]) # sum of the above 3 variables
      self.discountedCashFlowContainer.append(self.netIncomeContainer[-1]) #same as net income
      self.CumDCFContainer.append(self.netIncomeContainer[-1]) # same as net income
      return

    #A function to set up the following years making a nice
    def runYearly(self, year):
      self.fixedCostContainter.append(-calculateYearlyFixed(self) * calculateInflation(self,year)) #same calculation as year 0
      self.variableCostContainer.append(-calculateYearlyCosts(self) * calculateInflation(self,year)) # sum of all costs per product * the amount of product made
      self.revenueContainer.append(calculateYearlyRev(self) * calculateInflation(self,year)) #Selling price * the amount of product made
      self.operatingIncomeContainer.append(self.fixedCostContainter[-1] + self.variableCostContainer[-1] + self.revenueContainer[-1]) # sum of the above 3 variable
      self.netIncomeContainer.append((1-self.corporateTax) * self.operatingIncomeContainer[-1]) # epic
      self.discountedCashFlowContainer.append(npf.pv(rate = self.WACC, nper = year, pmt = 0, fv = -self.netIncomeContainer[-1])) # calculates the discounted cashflow using the numpy method for Present Value
      self.CumDCFContainer.append(self.CumDCFContainer[-1] + self.discountedCashFlowContainer[-1]) # previous CDCF + Discounted cashflow
      return

  #Acutal logic
    setupInitialState(self)
    for i in range(1,self.lifetime+1):
      runYearly(self, i)

    self.IRR = npf.irr(self.netIncomeContainer)
    self.NPV = npf.npv(self.WACC, self.netIncomeContainer)

    self.breakeven = calculateBreakeven(self)
    if self.printstuff:
      print(f"The Internal Rate of Return is {round(self.IRR * 100,2)}%")
      print(f"The NetPresentValue is ${round(self.NPV,2)}")
      if self.breakeven:
        print(f"the Breakevenpoint is after {round(self.breakeven,2)} years")
      else:
        print("There is no breakeven point")
    return

  def makeGraph(self):
    fig = go.Figure()

    fig.add_trace(go.Scatter(x=[x for x in range(0,self.lifetime+1)], y=self.fixedCostContainter, name="Fixed Costs"))
    fig.add_trace(go.Scatter(x=[x for x in range(0,self.lifetime+1)], y=self.variableCostContainer, name="Variable Costs"))
    fig.add_trace(go.Scatter(x=[x for x in range(0,self.lifetime+1)], y=self.revenueContainer, name="Revenue"))
    fig.add_trace(go.Scatter(x=[x for x in range(0,self.lifetime+1)], y=self.operatingIncomeContainer, name="Operating Income"))
    fig.add_trace(go.Scatter(x=[x for x in range(0,self.lifetime+1)], y=self.netIncomeContainer, name="Net Income"))
    fig.add_trace(go.Scatter(x=[x for x in range(0,self.lifetime+1)], y=self.discountedCashFlowContainer, name="Discounted Cash Flow"))
    fig.add_trace(go.Scatter(x=[x for x in range(0,self.lifetime+1)], y=self.CumDCFContainer, name="Cummulative DCF"))

    fig.show()
    return

  def getData(self):
    TotalDict = {
        "FixedCosts" : self.fixedCostContainter
        ,"VariableCosts" : self.variableCostContainer
        ,"Revenue" : self.revenueContainer
        ,"OperatingIncome" : self.operatingIncomeContainer
        ,"NetIncome" : self.netIncomeContainer
        ,"DiscountedCashFlow" : self.discountedCashFlowContainer
        ,"CummulativeDCF" : self.CumDCFContainer
        ,"InternalRateOfReturn" : self.IRR
        ,"NetPresentValue" : self.NPV
        ,"Breakeven" : self.breakeven
      }
    return TotalDict

  def getnpv(self):
    return self.NPV



In [ ]:
# @title Barebones Analysis
def addDefaultBreweryTest2():
  #redefine dicts
  #substanceDict = {}
  #costDict = {}
  #systemVariables = {}

  #adding the actual things
  CAPEXwithoutHEN = 604100
  CAPEXwithHEN = 747300
  costDict["Retrofitting HEN"] = {'type': 'once', 'costs': CAPEXwithHEN - CAPEXwithoutHEN,'substanceKey':'', 'category': 'fixed'}

  totalCostWithoutHEN = 40370000
  totalCostWithHEN = 31360000
  costDict['Total Costs'] = {'type': 'yearly', 'costs': totalCostWithHEN - totalCostWithoutHEN, 'substanceKey':'', 'category':'variable'}

  #adding systemvariables
  systemVariables['inflation'] = 0.035
  systemVariables['interest'] = 0.06
  systemVariables['beta'] = 0.79
  systemVariables['equityRiskPremium'] = 0.055
  systemVariables['equityRatio'] = 1
  systemVariables['lifetime'] = 20
  systemVariables['corpTax'] = 0.19
  systemVariables['printstuff'] = True

addDefaultBreweryTest2()

pog = companyToBeAnalysed(substanceDict, costDict, systemVariables)

#below this should appear a small indication of whether the Retrofit should be profitable, leaving out all other company variables
#it conconciders only the total costs as well as the initial costs.

The Internal Rate of Return is 5096.44%
The NetPresentValue is $78633581.35
the Breakevenpoint is after 0.02 years


In [38]:
#@title no HEN COMPANY
def addActualBreweryTest():
  #adding systemvariables
  systemVariables['inflation'] = 0.035
  systemVariables['interest'] = 0.06
  systemVariables['beta'] = 0.79
  systemVariables['equityRiskPremium'] = 0.055
  systemVariables['equityRatio'] = 1
  systemVariables['lifetime'] = 20
  systemVariables['corpTax'] = 0.19
  systemVariables['utilisationFactor'] = 0.9
  systemVariables["printstuff"] = True

  # adding substances
  maxBeer = 410204916.91
  beerProduced = maxBeer * systemVariables["utilisationFactor"]
  waterUsed = beerProduced * 0.95
  barleyUsed = beerProduced *0.2
  hopUsed = beerProduced * 0.002
  yeastUsed = beerProduced * 0.0005

  substanceDict["beer"] = {'input': 0, 'output': beerProduced, 'net': -beerProduced, 'type': 'product', 'cost': 1.22876068376068}
  substanceDict["water"] = {'input': waterUsed, 'output': 0, 'net': waterUsed, 'type': 'starting material', 'cost': 0.00145}
  substanceDict["barley"] = {'input': barleyUsed, 'output': 0, 'net': barleyUsed, 'type': 'starting material', 'cost': 2.08}
  substanceDict["hop"] = {'input': hopUsed, 'output': 0, 'net': hopUsed, 'type': 'starting material', 'cost': 36.344}
  substanceDict["yeast"] = {'input': yeastUsed, 'output': 0, 'net': yeastUsed, 'type': 'starting material', 'cost': 140.2173913}

  #adding costs (initial costs)
  costDict["basic install costs watersupply"] = {'type': 'once', 'costs': 1708,'substanceKey':'', 'category': 'fixed'}
  #-----\           604100
  costDict["CAPEX HEN"] = {'type': 'once', 'costs': 40000000,'substanceKey':'', 'category': 'fixed'}
  #-----/
  costDict["initial location costs"] = {'type': 'once', 'costs': 41760,'substanceKey':'', 'category': 'fixed'}
  #adding costs (yearly)
  costDict["24/7 watersuply"] = {'type':'yearly', 'costs':181.11, 'substanceKey':'', 'category':'fixed'}
  costDict["hiring Location"] = {'type': 'yearly', 'costs': 138000,'substanceKey':'', 'category': 'fixed'}
  costDict["total energy price"] = {'type': 'yearly', 'costs': 40277485.79,'substanceKey':'', 'category': 'variable'}
  costDict["Maintenance Cost"] = {'type': 'yearly', 'costs': 4000000,'substanceKey':'', 'category': 'fixed'}
  #-----\
  #costDict["total cost index"] = {'type': 'yearly', 'costs': 40370000,'substanceKey':'', 'category': 'fixed'}
  #-----/
  costDict["fixed supply rate for fixed and variable contracts"] = {'type': 'yearly', 'costs': 112.95,'substanceKey':'', 'category': 'fixed'}

  costDict['CO2 cost'] = {'type':"yearly", 'costs': 2821702.71, 'substanceKey':'', 'category': 'variable'}
  #adding substancedependent costs

  costDict["bottles"] = {'type': 'yearly per substance', 'costs': 0.55,'substanceKey':'beer', 'category': 'variable'}
  costDict["tax on water"] = {'type': 'yearly per substance', 'costs': 0.000437,'substanceKey':'water','category': 'variable'}

  #costDict["tax on electricity"] = {'type': 'yearly per substance', 'costs': 0.14,'substanceKey':'electricity','category': 'variable'}
  #costDict["electricity tax refund"] = {'type': 'yearly', 'costs': -635.19,'substanceKey':'', 'category': 'fixed'}
  #substanceDict["electricity"] = {'input': 146998288.896, 'output':0, 'net':146998288.896, "type":'starting material', 'cost':0.14}
  #costDict["Electricity transport rate"] = {'type': 'yearly', 'costs': 476.87,'substanceKey':'', 'category': 'fixed'}

addActualBreweryTest()

brewery = companyToBeAnalysed(substanceDict, costDict, systemVariables)
a = brewery.getData()
for key in a.keys():
  print(key)
  print(a[key])

The Internal Rate of Return is nan%
The NetPresentValue is $-71610705.91
There is no breakeven point


FixedCosts
[-40043468, -4138294.06, -4138294.06, -4138294.06, -4138294.06, -4138294.06, -4138294.06, -4138294.06, -4138294.06, -4138294.06, -4138294.06, -4138294.06, -4138294.06, -4138294.06, -4138294.06, -4138294.06, -4138294.06, -4138294.06, -4138294.06, -4138294.06, -4138294.06]
VariableCosts
[0, -453111477.72814065, -453111477.72814065, -453111477.72814065, -453111477.72814065, -453111477.72814065, -453111477.72814065, -453111477.72814065, -453111477.72814065, -453111477.72814065, -453111477.72814065, -453111477.72814065, -453111477.72814065, -453111477.72814065, -453111477.72814065, -453111477.72814065, -453111477.72814065, -453111477.72814065, -453111477.72814065, -453111477.72814065, -453111477.72814065]
Revenue
[0, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 4

In [39]:
#@title HEN COMPANY
def addNewBreweryTest():
  #adding systemvariables
  systemVariables['inflation'] = 0.035
  systemVariables['interest'] = 0.06
  systemVariables['beta'] = 0.79
  systemVariables['equityRiskPremium'] = 0.055
  systemVariables['equityRatio'] = 1
  systemVariables['lifetime'] = 20
  systemVariables['corpTax'] = 0.19
  systemVariables['utilisationFactor'] = 0.9
  systemVariables["printstuff"] = True

  # adding substances
  maxBeer = 410204916.91
  beerProduced = maxBeer * systemVariables["utilisationFactor"]
  waterUsed = beerProduced * 0.95
  barleyUsed = beerProduced *0.2
  hopUsed = beerProduced * 0.002
  yeastUsed = beerProduced * 0.0005

  substanceDict["beer"] = {'input': 0, 'output': beerProduced, 'net': -beerProduced, 'type': 'product', 'cost': 1.22876068376068}
  substanceDict["water"] = {'input': waterUsed, 'output': 0, 'net': waterUsed, 'type': 'starting material', 'cost': 0.00145}
  substanceDict["barley"] = {'input': barleyUsed, 'output': 0, 'net': barleyUsed, 'type': 'starting material', 'cost': 2.08}
  substanceDict["hop"] = {'input': hopUsed, 'output': 0, 'net': hopUsed, 'type': 'starting material', 'cost': 36.344}
  substanceDict["yeast"] = {'input': yeastUsed, 'output': 0, 'net': yeastUsed, 'type': 'starting material', 'cost': 140.2173913}

  #adding costs (initial costs)
  costDict["basic install costs watersupply"] = {'type': 'once', 'costs': 1708,'substanceKey':'', 'category': 'fixed'}
  #-----\           604100
  costDict["CAPEX HEN"] = {'type': 'once', 'costs': 40143200,'substanceKey':'', 'category': 'fixed'}
  #-----/
  costDict["initial location costs"] = {'type': 'once', 'costs': 41760,'substanceKey':'', 'category': 'fixed'}
  #adding costs (yearly)
  costDict["24/7 watersuply"] = {'type':'yearly', 'costs':181.11, 'substanceKey':'', 'category':'fixed'}
  costDict["hiring Location"] = {'type': 'yearly', 'costs': 138000,'substanceKey':'', 'category': 'fixed'}
  costDict["total energy price"] = {'type': 'yearly', 'costs': 31252413.57,'substanceKey':'', 'category': 'variable'}
  costDict["Maintenance Cost"] = {'type': 'yearly', 'costs': 4014320,'substanceKey':'', 'category': 'fixed'}
  #-----\
  #costDict["total cost index"] = {'type': 'yearly', 'costs': 40370000,'substanceKey':'', 'category': 'fixed'}
  #-----/
  costDict["fixed supply rate for fixed and variable contracts"] = {'type': 'yearly', 'costs': 112.95,'substanceKey':'', 'category': 'fixed'}

  costDict['CO2 cost'] = {'type':"yearly", 'costs': 2189437.71, 'substanceKey':'', 'category': 'variable'}
  #adding substancedependent costs

  costDict["bottles"] = {'type': 'yearly per substance', 'costs': 0.55,'substanceKey':'beer', 'category': 'variable'}
  costDict["tax on water"] = {'type': 'yearly per substance', 'costs': 0.000437,'substanceKey':'water','category': 'variable'}

  #costDict["tax on electricity"] = {'type': 'yearly per substance', 'costs': 0.14,'substanceKey':'electricity','category': 'variable'}
  #costDict["electricity tax refund"] = {'type': 'yearly', 'costs': -635.19,'substanceKey':'', 'category': 'fixed'}
  #substanceDict["electricity"] = {'input': 146998288.896, 'output':0, 'net':146998288.896, "type":'starting material', 'cost':0.14}
  #costDict["Electricity transport rate"] = {'type': 'yearly', 'costs': 476.87,'substanceKey':'', 'category': 'fixed'}

addNewBreweryTest()

brewery = companyToBeAnalysed(substanceDict, costDict, systemVariables)
a = brewery.getData()
for key in a.keys():
  print(key)
  print(a[key])

The Internal Rate of Return is 10.51%
The NetPresentValue is $12557510.18
the Breakevenpoint is after 12.42 years


FixedCosts
[-40186668, -4152614.06, -4152614.06, -4152614.06, -4152614.06, -4152614.06, -4152614.06, -4152614.06, -4152614.06, -4152614.06, -4152614.06, -4152614.06, -4152614.06, -4152614.06, -4152614.06, -4152614.06, -4152614.06, -4152614.06, -4152614.06, -4152614.06, -4152614.06]
VariableCosts
[0, -443454140.5081407, -443454140.5081407, -443454140.5081407, -443454140.5081407, -443454140.5081407, -443454140.5081407, -443454140.5081407, -443454140.5081407, -443454140.5081407, -443454140.5081407, -443454140.5081407, -443454140.5081407, -443454140.5081407, -443454140.5081407, -443454140.5081407, -443454140.5081407, -443454140.5081407, -443454140.5081407, -443454140.5081407, -443454140.5081407]
Revenue
[0, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 453639306.7658921, 45

In [40]:
# @title Both Company
addActualBreweryTest()
systemVariables["printstuff"] = False

brewery = companyToBeAnalysed(substanceDict, costDict, systemVariables)

oldbrewery = brewery.getData()

addNewBreweryTest()
systemVariables["printstuff"] = False

newbrewery = companyToBeAnalysed(substanceDict, costDict, systemVariables)

new = newbrewery.getData()

fig = go.Figure()

for keys in oldbrewery.keys():
  if keys not in ["InternalRateOfReturn","NetPresentValue","Breakeven","Revenue","VariableCosts","FixedCosts"]:
    fig.add_trace(go.Scatter(x=[x for x in range(0,21)], y=oldbrewery[keys], name=f"{keys} without HEN"))
    fig.add_trace(go.Scatter(x=[x for x in range(0,21)], y=new[keys], name=f"{keys} with HEN"))

fig.update_layout(title=dict(text="Techno-Economical Analysis"), xaxis=dict(title=dict(text="Years of operation")), yaxis=dict(title=dict(text="Money(Euro's)")))
fig.show()

In [34]:
# @title Sensitivity Analysis
fig = go.Figure()

parameters = [('system','inflation'), ('system','interest'), ('system','beta'),
              ('substance', 'beer', 'cost'),('substance', 'water', 'cost'),
              ('substance', 'barley', 'cost'),('substance', 'hop', 'cost'),('substance', 'yeast', 'cost'),
              ('cost', 'bottles', 'costs'), ("cost","Maintenance Cost","costs"), ("cost","total energy price","costs")]


addNewBreweryTest()
#parameters = [('system','inflation')]
systemVariables["printstuff"] = False
dictDict = {'substance':substanceDict, 'cost':costDict,'system':systemVariables}
for items in parameters:
  valuelist = []
  npvresults = []
  if len(items) == 2: actualvalue = dictDict[items[0]][items[1]]
  else: actualvalue = dictDict[items[0]][items[1]][items[2]]
  for value in [(actualvalue + actualvalue * (x/100))  for x in range(-20,21)]:
    valuelist.append(value)
    if len(items) == 2: dictDict[items[0]][items[1]] = value
    else: dictDict[items[0]][items[1]][items[2]] = value
    brewery = companyToBeAnalysed(dictDict["substance"], dictDict["cost"], dictDict["system"])
    npvresults.append(brewery.getnpv())
  if len(items) == 2:
    dictDict[items[0]][items[1]] = actualvalue
    fig.add_trace(go.Scatter(x=[f"{x}%" for x in range(-20,21)], y=npvresults, name=items[1]))
  else:
    dictDict[items[0]][items[1]][items[2]] = actualvalue
    fig.add_trace(go.Scatter(x=[f"{x}%" for x in range(-20,21)], y=npvresults, name=f"{items[1]} {items[2]}"))

fig.update_layout(title=dict(text="Sensitivity Analysis"), xaxis=dict(title=dict(text="Percentage change in variable")), yaxis=dict(title=dict(text="NPV after 20 years")))
fig.show()

In [31]:
from numpy._core.defchararray import add
# @title prediction based on inflation and tax
def makefancyGraph(name, rgb):
  dictDict = {'substance':substanceDict, 'cost':costDict,'system':systemVariables}
  basecaseData = companyToBeAnalysed(substanceDict, costDict, systemVariables).getData()
  transformedData = [[] for x in range(21)]

  inflationNumber = systemVariables["inflation"]

  energyprice = costDict["total energy price"]["costs"]
  #beerprice = substanceDict["beer"]["cost"]

  #watertax = costDict["tax on water"]["costs"]
  #electricitytax = costDict["tax on electricity"]["costs"]
  #CO2tax = costDict["CO2 tax"]["costs"]
  inflationRange = [np.random.normal(0.02, 0.015) for x in range(10000)]

  #normalBeerRange = [np.random.normal(beerprice, beerprice*0.0025) for x in range(100)]

  for inf in inflationRange:
    #for ep in [(energyprice + energyprice * (x/100))  for x in range(-5,6)]:
      #for bp in normalBeerRange:
        #for ct in [(CO2tax + CO2tax * (x/100))  for x in range(-5,6)]:
          dictDict["system"]["inflation"] = inf
          #dictDict["cost"]["total energy price"]["costs"] = ep
          #dictDict["substance"]["beer"]["cost"] = bp
          #dictDict["cost"]["CO2 tax"]["costs"] = ct
          brewerydata = companyToBeAnalysed(dictDict["substance"], dictDict["cost"], dictDict["system"]).getData()
          for i in range(len(transformedData)):
            transformedData[i].append(brewerydata["CummulativeDCF"][i])

  dictDict["system"]["inflation"] = inflationNumber
  #dictDict["cost"]["tax on water"]["costs"] = watertax
  #dictDict["cost"]["tax on electricity"]["costs"] = electricitytax
  #dictDict["cost"]["CO2 tax"]["costs"] = CO2tax



  c95 = []
  c75 = []
  c55 = []
  c50 = []
  c45 = []
  c25 = []
  c5  = []
  mean = []

  for i in range(len(transformedData)):
    c95.append(np.percentile(transformedData[i],95))
    c75.append(np.percentile(transformedData[i],75))
    c55.append(np.percentile(transformedData[i],55))
    mean.append(np.mean(transformedData[i]))
    c45.append(np.percentile(transformedData[i],45))
    c25.append(np.percentile(transformedData[i],25))
    c5.append(np.percentile(transformedData[i],5))


  timeseries = [x for x in range(0,21)]

  #fig.add_trace(go.Scatter(x=timeseries, y=basecaseData["FixedCosts"], name="Fixed Costs"))
  #fig.add_trace(go.Scatter(x=timeseries, y=basecaseData["VariableCosts"], name="Variable Costs"))
  #fig.add_trace(go.Scatter(x=timeseries, y=basecaseData["Revenue"], name="Revenue"))
  #fig.add_trace(go.Scatter(x=timeseries, y=basecaseData["OperatingIncome"], name=f"Operating Income{name}"))
  fig.add_trace(go.Scatter(x=timeseries, y=basecaseData["NetIncome"], name=f"Net Income{name}"))
  fig.add_trace(go.Scatter(x=timeseries, y=basecaseData["DiscountedCashFlow"], name=f"Discounted Cash Flow{name}"))
  fig.add_trace(go.Scatter(x=timeseries, y=mean, name=f"Mean CummulativeDCF{name}", line=dict(color=f'rgba({rgb},1)')))

  fig.add_trace(go.Scatter(
      x=timeseries + timeseries[::-1],
      y=c95 + c5[::-1],
      fill='toself',
      fillcolor=f'rgba({rgb},0.15)',
      line=dict(color='rgba(255,255,255,0)'),
      name=f'90% range - CummulativeDCF{name}'
  ))
  fig.add_trace(go.Scatter(
      x=timeseries + timeseries[::-1],
      y=c75 + c25[::-1],
      fill='toself',
      fillcolor=f'rgba({rgb},0.3)',
      line=dict(color='rgba(255,255,255,0)'),
      name=f'50% range - CummulativeDCF{name}'
  ))
  fig.add_trace(go.Scatter(
      x=timeseries + timeseries[::-1],
      y=c55 + c45[::-1],
      fill='toself',
      fillcolor=f'rgba({rgb},0.4)',
      line=dict(color='rgba(255,255,255,0)'),
      name=f'10% range - CummulativeDCF{name}'
  ))

fig = go.Figure()

addNewBreweryTest()
systemVariables["printstuff"] = False
makefancyGraph(" with HEN", '100, 255, 0')

addActualBreweryTest()
systemVariables["printstuff"] = False
makefancyGraph(" without HEN", '0,100,255')




fig.update_layout(title=dict(text="Expected returns based on varying Inflation (1.5% to 3.5%, normally distributed)"), xaxis=dict(title=dict(text="Operating years")), yaxis=dict(title=dict(text="Money (Euro's)")))

fig.show()